# RMT-PPAD migration NB84 - Phase P5 (lane_only_mode in MTDETRDLoss)

**Purpose.** Verify that running `model.model.loss(batch)` on a synthetic
lane-only batch produces finite per-term losses, that `da_seg == 0.0`
exactly (drivable losses removed), and that the 4 lane keys
(`lane_cls_loss, lane_xytl_loss, lane_iou_loss, lane_seg_aux_loss`)
appear in the per-step loss dict.

**Acceptance (appendix-path3 sec 8.7):** synthetic batch produces finite
losses; `da_seg == 0.0`; backward pass succeeds.

**Wall time:** ~2-3 min (mmcv install + model build + loss+backward).

### Cell 1: Mount Drive, install mmcv

In [1]:
import os, sys, subprocess
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import mmcv  # noqa: F401
    print(f'[ok] mmcv already installed: {mmcv.__version__}')
except ImportError:
    print('[install] mmcv (~1-3 min wheel build)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv  # noqa: F401
    print(f'[ok] mmcv installed: {mmcv.__version__}')

import torch
print('torch', torch.__version__)
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
[install] mmcv (~1-3 min wheel build)...
[ok] mmcv installed: 2.2.0
torch 2.10.0+cpu
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


### Cell 2: Smoke test the lane_losses utilities (line_iou + assign + FocalLossForLane)

In [2]:
import sys, os
SCRIPT = 'stage2/rmt_ppad_migration/P5_loss/tools/lane_losses.py'
log = os.path.join(LOG_DIR, 'NB84_lane_losses_smoke.log')
rc = run_streaming([sys.executable, '-u', SCRIPT], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'lane_losses smoke rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P5_loss/tools/lane_losses.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB84_lane_losses_smoke.log
[smoke] lane_losses starting
[smoke] line_iou(identical): 1.000  (expect 1.000)
[smoke] pair-wise IoU shape: (5, 3)  (expect (5, 3))
[smoke] liou_loss(identical): 0.0000  (expect 0)
[smoke] assign: matched 1 priors to 3 GTs
[smoke] FocalLossForLane shape=(192,) sum=21.662
[smoke] PASS
[run_streaming] return_code=0


### Cell 3: Build clr_lane model + run loss on synthetic batch
Asserts: `MTDETRDLoss.lane_only_mode=True`, `da_seg==0.0`, finite,
backward OK.

In [3]:
import sys, os
VERIFY = 'stage2/rmt_ppad_migration/P5_loss/tools/verify_lane_loss.py'
log = os.path.join(LOG_DIR, 'NB84_verify_lane_loss.log')
rc = run_streaming([sys.executable, '-u', VERIFY], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'verify_lane_loss rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P5_loss/tools/verify_lane_loss.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB84_verify_lane_loss.log
[smoke] importing MTDETR from /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[run_streaming] still running; no child output yet. This usually means the first dataloader/model step is still working.
[smoke] building model from /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml
WARNING ⚠️ no model scale passed. Assuming scale='l'.
[

### Cell 4: In-process re-run + per-key loss dump
Same model + batch, run inside the notebook process so we can dump
the full per-key loss dictionary.

In [4]:
import sys, torch
from pathlib import Path

RMT_PPAD_ROOT = Path(REPO_ROOT) / 'stage2' / 'rmt_ppad_migration' / 'vendor' / 'RMT-PPAD'
YAML_PATH = RMT_PPAD_ROOT / 'ultralytics' / 'cfg' / 'models' / 'mt-detr' / 'rtdetr-l_bdd_clr_lane.yaml'

# Force vendored ultralytics.
sys.path = [x for x in sys.path if 'ultralytics' not in x.lower()]
sys.path.insert(0, str(RMT_PPAD_ROOT))
for k in list(sys.modules):
    if k == 'ultralytics' or k.startswith('ultralytics.'):
        del sys.modules[k]

from ultralytics import MTDETR
TOOLS_P5 = str(Path(REPO_ROOT) / 'stage2' / 'rmt_ppad_migration' / 'P5_loss' / 'tools')
if TOOLS_P5 not in sys.path:
    sys.path.insert(0, TOOLS_P5)
from verify_lane_loss import _build_synthetic_batch

model = MTDETR(str(YAML_PATH))
inner = model.model.cpu().train()
if not hasattr(inner, 'criterion') or inner.criterion is None:
    inner.criterion = inner.init_criterion()
print('criterion:', type(inner.criterion).__name__,
      '| lane_only_mode =', inner.criterion.lane_only_mode)

batch = _build_synthetic_batch(B=2, img_size=640, max_lanes=8)
loss_terms, loss_items, _ = inner.loss(batch)
L_det, da_seg, ll_seg = loss_terms
print(f'\nL_det  = {float(L_det):.4f}')
print(f'da_seg = {float(da_seg):.4f}  (expect exactly 0)')
print(f'll_seg = {float(ll_seg):.4f}')

# Re-run forward but call the criterion directly so we get per-key dict.
img = batch['img']
type_task = batch['type_task'][0]
import torch
detection_classes = torch.tensor(type_task['detection'])
detection_indices = torch.where(torch.isin(batch['cls'], detection_classes))[0]
gt_groups = [(batch['batch_idx'][detection_indices] == i).sum().item() for i in range(2)]
targets_det = {
    'cls': batch['cls'][detection_indices].long().view(-1),
    'bboxes': batch['bboxes'][detection_indices],
    'batch_idx': batch['batch_idx'][detection_indices].long().view(-1),
    'gt_groups': gt_groups,
}
preds = inner.predict(img, batch=targets_det)
dec_bboxes, dec_scores, enc_bboxes, enc_scores, dn_meta, seg_masks, _aux = preds
if dn_meta is None:
    dn_bboxes, dn_scores = None, None
else:
    dn_bboxes, dec_bboxes = torch.split(dec_bboxes, dn_meta['dn_num_split'], dim=2)
    dn_scores, dec_scores = torch.split(dec_scores, dn_meta['dn_num_split'], dim=2)
dec_bboxes = torch.cat([enc_bboxes.unsqueeze(0), dec_bboxes])
dec_scores = torch.cat([enc_scores.unsqueeze(0), dec_scores])
targets_seg = {
    'lane_seg_mask': batch['lane_seg_mask'],
    'lane_targets':  batch['lane_targets'],
}
per_key = inner.criterion(
    (dec_bboxes, dec_scores), targets_det,
    dn_bboxes=dn_bboxes, dn_scores=dn_scores, dn_meta=dn_meta,
    seg_mask=[seg_masks, None], seg_batch=targets_seg,
)
print('\nPer-key loss dict:')
for k in sorted(per_key):
    v = per_key[k]
    val = float(v) if hasattr(v, 'item') else v
    marker = '  <-- LANE' if k.startswith('lane_') else ''
    print(f'  {k:30s} {val:.4f}{marker}')

# Sanity assertions.
assert float(da_seg) == 0.0, f'da_seg must be 0, got {float(da_seg)}'
lane_keys = ['lane_cls_loss', 'lane_xytl_loss', 'lane_iou_loss', 'lane_seg_aux_loss']
for k in lane_keys:
    assert k in per_key, f'missing key {k}'
print('\n[P5 result] PASS')

WARNING ⚠️ no model scale passed. Assuming scale='l'.
criterion: MTDETRDLoss | lane_only_mode = True


/tmp/ipykernel_1596/4010564616.py:30: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f'\nL_det  = {float(L_det):.4f}')



L_det  = 495.8939
da_seg = 0.0000  (expect exactly 0)
ll_seg = 24.5635

Per-key loss dict:
  lane_cls_loss                  16.6029  <-- LANE
  lane_iou_loss                  2.0000  <-- LANE
  lane_seg_aux_loss              0.9395  <-- LANE
  lane_xytl_loss                 4.9635  <-- LANE
  loss_bbox                      2.0625
  loss_bbox_aux                  11.7500
  loss_bbox_aux_dn               6.4284
  loss_bbox_dn                   1.2857
  loss_class                     54.4026
  loss_class_aux                 395.3726
  loss_class_aux_dn              1.6779
  loss_class_dn                  0.4203
  loss_giou                      2.0493
  loss_giou_aux                  12.5821
  loss_giou_aux_dn               7.0502
  loss_giou_dn                   1.4100

[P5 result] PASS
